In [2]:
# ============================================================
# CELL 1 — INSTALL DEPENDENCIES
# ============================================================

!pip install -q groq tavily-python arxiv gradio beautifulsoup4 pymupdf

# Keep Google Colab's required requests version
!pip install -q requests==2.32.4

print("✅ Dependencies installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.8/64.8 kB 4.4 MB/s eta 0:00:00
✅ Dependencies installed.


In [6]:
from google.colab import userdata

print("Testing Colab Secrets...\n")

try:
    groq_key = userdata.get("GROQ_API_KEY")

    if groq_key:
        print("✅ GROQ_API_KEY found")
        print("   Starts with:", groq_key[:4] + "...")
        print("   Length:", len(groq_key))
    else:
        print("❌ GROQ_API_KEY is empty")

except Exception as e:
    print("❌ GROQ_API_KEY could not be accessed")
    print("Error:", e)


try:
    tavily_key = userdata.get("TAVILY_API_KEY")

    if tavily_key:
        print("✅ TAVILY_API_KEY found")
        print("   Starts with:", tavily_key[:5] + "...")
        print("   Length:", len(tavily_key))
    else:
        print("❌ TAVILY_API_KEY is empty")

except Exception as e:
    print("❌ TAVILY_API_KEY could not be accessed")
    print("Error:", e)

Testing Colab Secrets...

✅ GROQ_API_KEY found
   Starts with: gsk_...
   Length: 56
✅ TAVILY_API_KEY found
   Starts with: tvly-...
   Length: 58


In [7]:
# ============================================================
# CELL 2 — LOAD API KEYS + INITIALIZE APIs
# ============================================================

from google.colab import userdata

# Load API keys
GROQ_API_KEY = userdata.get("GROQ_API_KEY")
TAVILY_API_KEY = userdata.get("TAVILY_API_KEY")

# Validate
if not GROQ_API_KEY:
    raise ValueError("❌ GROQ_API_KEY not found.")

if not TAVILY_API_KEY:
    raise ValueError("❌ TAVILY_API_KEY not found.")

# Initialize Groq
from groq import Groq

groq_client = Groq(
    api_key=GROQ_API_KEY
)

# Initialize Tavily
from tavily import TavilyClient

tavily_client = TavilyClient(
    api_key=TAVILY_API_KEY
)

# Model
GROQ_MODEL = "openai/gpt-oss-20b"

# Research configuration
MAX_SUBQUESTIONS = 3
MAX_TAVILY_RESULTS_PER_QUERY = 3
MAX_ARXIV_RESULTS_PER_QUERY = 1
ARXIV_DELAY_SECONDS = 3

print("==============================================")
print("       AGENTIC RESEARCH ASSISTANT")
print("==============================================")
print("✅ GROQ_API_KEY loaded")
print("✅ TAVILY_API_KEY loaded")
print("✅ Groq client initialized")
print("✅ Tavily client initialized")
print(f"✅ Groq model: {GROQ_MODEL}")
print("==============================================")

       AGENTIC RESEARCH ASSISTANT
✅ GROQ_API_KEY loaded
✅ TAVILY_API_KEY loaded
✅ Groq client initialized
✅ Tavily client initialized
✅ Groq model: openai/gpt-oss-20b


In [8]:
# ============================================================
# CELL 3 — PLANNER AGENT
# ============================================================

def planner_agent(user_query: str):
    """
    Convert the user's research question into exactly
    3 focused research sub-questions.
    """

    if not user_query or not user_query.strip():
        raise ValueError("Research question cannot be empty.")

    prompt = f"""
You are the planning agent of an Agentic Research Assistant.

The user wants to research this topic:

"{user_query}"

Create exactly 3 focused research sub-questions.

Requirements:
1. The questions must be directly related to the user's topic.
2. Dynamically understand the actual topic.
3. Cover different important dimensions.
4. Avoid duplicate questions.
5. Make them suitable for web and academic research.
6. Return ONLY the numbered questions.
7. Do not provide explanations.

Example format:

1. What are the major aspects of this topic?
2. What evidence and recent developments exist?
3. What are the major challenges, impacts, and future directions?
"""

    try:
        response = groq_client.chat.completions.create(
            model=GROQ_MODEL,
            messages=[
                {
                    "role": "system",
                    "content": "You are a precise research planning agent."
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            temperature=0.2,
            max_tokens=500
        )

        text = response.choices[0].message.content.strip()

        questions = []

        for line in text.splitlines():
            line = line.strip()

            match = re.match(r"^\d+[\.\)]\s*(.+)$", line)

            if match:
                question = match.group(1).strip()

                if question:
                    questions.append(question)

        # Fallback if the model doesn't return 3 questions
        if len(questions) < 3:
            questions = [
                f"What are the main aspects of {user_query}?",
                f"What evidence and recent developments exist regarding {user_query}?",
                f"What are the major challenges, impacts, and future directions related to {user_query}?"
            ]

        return questions[:3]

    except Exception as e:
        print("⚠️ Planner encountered an error:")
        print(e)

        return [
            f"What are the main aspects of {user_query}?",
            f"What evidence and recent developments exist regarding {user_query}?",
            f"What are the major challenges, impacts, and future directions related to {user_query}?"
        ]


print("✅ Planner Agent ready.")

✅ Planner Agent ready.


In [9]:
# ============================================================
# CELL 4 — TAVILY WEB SEARCH
# ============================================================

def search_tavily(query: str, max_results: int = 3):
    """
    Search the web using Tavily and return
    normalized source information.
    """

    if not query or not query.strip():
        return []

    try:
        response = tavily_client.search(
            query=query,
            search_depth="basic",
            max_results=max_results,
            include_answer=False,
            include_raw_content=False
        )

        results = response.get("results", [])

        sources = []

        for result in results:
            title = str(result.get("title", "")).strip()
            url = str(result.get("url", "")).strip()
            content = str(result.get("content", "")).strip()

            if not url:
                continue

            sources.append({
                "source_type": "web",
                "title": title or "Untitled Web Source",
                "url": url,
                "content": content,
                "query": query
            })

        return sources

    except Exception as e:
        print(f"⚠️ Tavily search failed for: {query}")
        print(f"Error: {e}")
        return []


print("✅ Tavily search function ready.")

✅ Tavily search function ready.


In [10]:
# ============================================================
# CELL 5 — arXiv ACADEMIC SEARCH
# ============================================================

import arxiv
import time


def search_arxiv(query: str, max_results: int = 1):
    """
    Search arXiv for academic papers.

    The function is designed to be conservative with requests
    to avoid arXiv rate-limit errors.
    """

    if not query or not query.strip():
        return []

    try:
        # Small delay between arXiv requests
        time.sleep(3)

        client = arxiv.Client(
            page_size=max_results,
            delay_seconds=3.0,
            num_retries=1
        )

        search = arxiv.Search(
            query=query,
            max_results=max_results,
            sort_by=arxiv.SortCriterion.Relevance
        )

        sources = []

        for paper in client.results(search):

            if not paper.pdf_url:
                continue

            sources.append({
                "source_type": "arxiv",
                "title": paper.title.strip(),
                "url": paper.pdf_url,
                "content": paper.summary.strip(),
                "summary": paper.summary.strip(),
                "authors": [
                    author.name
                    for author in paper.authors
                ],
                "published": (
                    paper.published.isoformat()
                    if paper.published
                    else ""
                ),
                "query": query
            })

        return sources

    except Exception as e:

        error_message = str(e)

        if "429" in error_message:
            print("⚠️ arXiv rate limit reached.")
            print("Continuing without arXiv results.")

        else:
            print("⚠️ arXiv search failed:")
            print(error_message)

        # Do not stop the complete research pipeline
        return []


print("✅ arXiv search function ready.")

✅ arXiv search function ready.


In [11]:
# ============================================================
# CELL 6 — SEARCHER AGENT
# ============================================================

def searcher_agent(subquestions):
    """
    Search Tavily and arXiv for each research sub-question.
    """

    all_sources = []

    if not subquestions:
        return all_sources

    for i, question in enumerate(subquestions, start=1):

        print()
        print("=" * 60)
        print(f"🔎 RESEARCHING SUB-QUESTION {i}/{len(subquestions)}")
        print("=" * 60)
        print(question)

        # ----------------------------------------------------
        # Tavily web search
        # ----------------------------------------------------

        web_sources = search_tavily(
            question,
            max_results=MAX_TAVILY_RESULTS_PER_QUERY
        )

        print(f"🌐 Tavily results: {len(web_sources)}")

        all_sources.extend(web_sources)

        # ----------------------------------------------------
        # arXiv academic search
        # ----------------------------------------------------

        academic_sources = search_arxiv(
            question,
            max_results=MAX_ARXIV_RESULTS_PER_QUERY
        )

        print(f"📚 arXiv results: {len(academic_sources)}")

        all_sources.extend(academic_sources)

    print()
    print("=" * 60)
    print(f"✅ TOTAL SOURCES FOUND: {len(all_sources)}")
    print("=" * 60)

    return all_sources


print("✅ Searcher Agent ready.")

✅ Searcher Agent ready.


In [12]:
# ============================================================
# CELL 7 — READER AGENT
# ============================================================

def reader_agent(raw_sources):
    """
    Normalize and clean all retrieved sources into a
    consistent structure for the next agents.
    """

    if not raw_sources:
        return []

    cleaned_sources = []
    seen_urls = set()

    for source in raw_sources:

        url = str(source.get("url", "")).strip()
        title = str(source.get("title", "")).strip()
        content = str(source.get("content", "")).strip()

        source_type = source.get(
            "source_type",
            "web"
        )

        # ----------------------------------------------------
        # Skip invalid sources
        # ----------------------------------------------------

        if not url:
            continue

        if url in seen_urls:
            continue

        if not content:
            continue

        # Ignore extremely short content
        if len(content) < 80:
            continue

        seen_urls.add(url)

        # ----------------------------------------------------
        # Create normalized source
        # ----------------------------------------------------

        normalized = {
            "source_type": source_type,
            "title": title or "Untitled Source",
            "url": url,
            "content": content,
            "query": source.get("query", "")
        }

        # ----------------------------------------------------
        # Preserve academic information
        # ----------------------------------------------------

        if source_type == "arxiv":

            normalized["summary"] = source.get(
                "summary",
                content
            )

            normalized["authors"] = source.get(
                "authors",
                []
            )

            normalized["published"] = source.get(
                "published",
                ""
            )

        cleaned_sources.append(normalized)

    print(f"📖 Reader processed {len(cleaned_sources)} sources.")

    return cleaned_sources


print("✅ Reader Agent ready.")

✅ Reader Agent ready.


In [24]:
# ============================================================
# CELL 8 — FINAL HYBRID CRITIC AGENT
# ============================================================

import re


def _llm_source_evaluation(
    source_title,
    source_type,
    research_query,
    content
):
    """
    Ask Groq to evaluate source quality.

    Returns:
        dict or None
    """

    prompt = f"""
You are a research source quality evaluator.

Research question:
{research_query}

Source title:
{source_title}

Source type:
{source_type}

Source content:
{content[:3000]}

Evaluate this source on a 0-5 scale:

1. Relevance
2. Credibility
3. Evidence quality
4. Usefulness

Then calculate an overall score from 0-10.

Return ONLY this exact format:

SCORE: 8
RELEVANCE: 5
CREDIBILITY: 4
EVIDENCE: 4
USEFULNESS: 5
REASON: Relevant and useful evidence for the research question.
"""

    try:

        response = groq_client.chat.completions.create(
            model=GROQ_MODEL,
            messages=[
                {
                    "role": "system",
                    "content": (
                        "You evaluate research sources "
                        "conservatively and accurately."
                    )
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            temperature=0.0,
            max_tokens=250
        )

        result = response.choices[0].message.content.strip()

        # ----------------------------------------------------
        # Extract values
        # ----------------------------------------------------

        score_match = re.search(
            r"SCORE\s*:\s*([0-9]+(?:\.[0-9]+)?)",
            result,
            re.IGNORECASE
        )

        relevance_match = re.search(
            r"RELEVANCE\s*:\s*([0-9]+(?:\.[0-9]+)?)",
            result,
            re.IGNORECASE
        )

        credibility_match = re.search(
            r"CREDIBILITY\s*:\s*([0-9]+(?:\.[0-9]+)?)",
            result,
            re.IGNORECASE
        )

        evidence_match = re.search(
            r"EVIDENCE\s*:\s*([0-9]+(?:\.[0-9]+)?)",
            result,
            re.IGNORECASE
        )

        usefulness_match = re.search(
            r"USEFULNESS\s*:\s*([0-9]+(?:\.[0-9]+)?)",
            result,
            re.IGNORECASE
        )

        reason_match = re.search(
            r"REASON\s*:\s*(.*)",
            result,
            re.IGNORECASE
        )

        if not score_match:
            return None

        return {
            "overall_score": max(
                0,
                min(
                    10,
                    float(score_match.group(1))
                )
            ),

            "relevance": max(
                0,
                min(
                    5,
                    float(
                        relevance_match.group(1)
                    )
                )
            ) if relevance_match else 0,

            "credibility": max(
                0,
                min(
                    5,
                    float(
                        credibility_match.group(1)
                    )
                )
            ) if credibility_match else 0,

            "evidence_quality": max(
                0,
                min(
                    5,
                    float(
                        evidence_match.group(1)
                    )
                )
            ) if evidence_match else 0,

            "usefulness": max(
                0,
                min(
                    5,
                    float(
                        usefulness_match.group(1)
                    )
                )
            ) if usefulness_match else 0,

            "reason": (
                reason_match.group(1).strip()
                if reason_match
                else "LLM evaluation completed."
            ),

            "evaluation_method": "LLM"
        }

    except Exception:
        return None


# ============================================================
# FALLBACK HEURISTIC EVALUATOR
# ============================================================

def _heuristic_source_evaluation(
    source,
    research_query
):
    """
    Deterministic fallback evaluator.

    Used when the LLM critic cannot evaluate a source.
    """

    title = source.get(
        "title",
        ""
    ).lower()

    content = source.get(
        "content",
        ""
    ).lower()

    source_type = source.get(
        "source_type",
        "web"
    ).lower()

    query_words = set(
        re.findall(
            r"\b[a-zA-Z]{4,}\b",
            research_query.lower()
        )
    )

    source_text = title + " " + content

    # --------------------------------------------------------
    # Relevance
    # --------------------------------------------------------

    matches = sum(
        1
        for word in query_words
        if word in source_text
    )

    if matches >= 8:
        relevance = 5

    elif matches >= 5:
        relevance = 4

    elif matches >= 3:
        relevance = 3

    elif matches >= 1:
        relevance = 2

    else:
        relevance = 1

    # --------------------------------------------------------
    # Credibility
    # --------------------------------------------------------

    credibility = 2

    if source_type == "arxiv":
        credibility = 5

    academic_terms = [
        "research",
        "study",
        "clinical",
        "journal",
        "peer",
        "systematic review",
        "scoping review",
        "paper",
        "analysis"
    ]

    if any(
        term in source_text
        for term in academic_terms
    ):
        credibility = max(
            credibility,
            4
        )

    trusted_domains = [
        ".gov",
        ".edu",
        ".org",
        "who.int",
        "nih.gov",
        "nature.com",
        "sciencedirect.com",
        "pubmed",
        "springer.com",
        "ieee.org",
        "acm.org"
    ]

    url = source.get(
        "url",
        ""
    ).lower()

    if any(
        domain in url
        for domain in trusted_domains
    ):
        credibility = max(
            credibility,
            4
        )

    # --------------------------------------------------------
    # Evidence quality
    # --------------------------------------------------------

    if len(content) >= 1500:
        evidence_quality = 4

    elif len(content) >= 700:
        evidence_quality = 3

    elif len(content) >= 300:
        evidence_quality = 2

    else:
        evidence_quality = 1

    if any(
        term in content
        for term in [
            "study",
            "results",
            "findings",
            "data",
            "clinical trial",
            "participants",
            "analysis"
        ]
    ):
        evidence_quality = min(
            5,
            evidence_quality + 1
        )

    # --------------------------------------------------------
    # Usefulness
    # --------------------------------------------------------

    usefulness = round(
        (
            relevance
            + credibility
            + evidence_quality
        ) / 1.5,
        1
    )

    usefulness = max(
        0,
        min(
            5,
            usefulness
        )
    )

    # --------------------------------------------------------
    # Overall score
    # --------------------------------------------------------

    overall_score = round(
        (
            relevance * 0.40
            + credibility * 0.30
            + evidence_quality * 0.20
            + usefulness * 0.10
        ) * 2,
        1
    )

    return {
        "overall_score": overall_score,
        "relevance": relevance,
        "credibility": credibility,
        "evidence_quality": evidence_quality,
        "usefulness": usefulness,
        "reason": (
            "Fallback evaluation based on source type, "
            "content quality, relevance indicators, "
            "and evidence signals."
        ),
        "evaluation_method": "Heuristic"
    }


# ============================================================
# MAIN CRITIC
# ============================================================

def critic_filter(
    sources,
    max_sources=8
):
    """
    Final hybrid critic.

    1. Removes duplicate URLs.
    2. Attempts LLM evaluation.
    3. Falls back to deterministic evaluation.
    4. Ranks all sources.
    5. Keeps the strongest sources.
    """

    if not sources:
        print(
            "⚠️ No sources available for critic."
        )
        return []

    # --------------------------------------------------------
    # Deduplicate
    # --------------------------------------------------------

    unique_sources = []
    seen_urls = set()

    for source in sources:

        url = source.get(
            "url",
            ""
        ).strip()

        if not url:
            continue

        if url in seen_urls:
            continue

        seen_urls.add(url)

        unique_sources.append(
            source
        )

    print(
        f"🔍 Evaluating "
        f"{len(unique_sources)} unique sources..."
    )

    evaluated_sources = []

    llm_count = 0
    fallback_count = 0

    # --------------------------------------------------------
    # Evaluate every source
    # --------------------------------------------------------

    for i, source in enumerate(
        unique_sources,
        start=1
    ):

        title = source.get(
            "title",
            "Untitled Source"
        )

        content = source.get(
            "content",
            ""
        )

        source_type = source.get(
            "source_type",
            "web"
        )

        query = source.get(
            "query",
            ""
        )

        # ----------------------------------------------------
        # Try LLM evaluation
        # ----------------------------------------------------

        evaluation = _llm_source_evaluation(
            source_title=title,
            source_type=source_type,
            research_query=query,
            content=content
        )

        # ----------------------------------------------------
        # Fallback if LLM fails
        # ----------------------------------------------------

        if evaluation is None:

            evaluation = _heuristic_source_evaluation(
                source,
                query
            )

            fallback_count += 1

            method_symbol = "🔧"

        else:

            llm_count += 1

            method_symbol = "🤖"

        # ----------------------------------------------------
        # Store evaluation
        # ----------------------------------------------------

        evaluated_source = dict(
            source
        )

        evaluated_source[
            "critic_score"
        ] = evaluation[
            "overall_score"
        ]

        evaluated_source[
            "critic_relevance"
        ] = evaluation[
            "relevance"
        ]

        evaluated_source[
            "critic_credibility"
        ] = evaluation[
            "credibility"
        ]

        evaluated_source[
            "critic_evidence_quality"
        ] = evaluation[
            "evidence_quality"
        ]

        evaluated_source[
            "critic_usefulness"
        ] = evaluation[
            "usefulness"
        ]

        evaluated_source[
            "critic_reason"
        ] = evaluation[
            "reason"
        ]

        evaluated_source[
            "critic_method"
        ] = evaluation[
            "evaluation_method"
        ]

        evaluated_sources.append(
            evaluated_source
        )

        print(
            f"   [{i}/{len(unique_sources)}] "
            f"{method_symbol} "
            f"{evaluation['overall_score']:.1f}/10 — "
            f"{title[:55]}"
        )

    # --------------------------------------------------------
    # Sort by quality
    # --------------------------------------------------------

    evaluated_sources.sort(
        key=lambda source: (
            source.get(
                "critic_score",
                0
            ),
            source.get(
                "critic_relevance",
                0
            ),
            source.get(
                "critic_credibility",
                0
            )
        ),
        reverse=True
    )

    # --------------------------------------------------------
    # Select sources
    #
    # Primary threshold:
    # score >= 5
    #
    # Relevance protection:
    # relevance >= 2
    # --------------------------------------------------------

    approved_sources = [
        source
        for source in evaluated_sources
        if (
            source.get(
                "critic_score",
                0
            ) >= 5
            and
            source.get(
                "critic_relevance",
                0
            ) >= 2
        )
    ]

    # --------------------------------------------------------
    # Limit number of sources
    # --------------------------------------------------------

    approved_sources = approved_sources[
        :max_sources
    ]

    # --------------------------------------------------------
    # Safety fallback
    #
    # If everything scores poorly, keep the best 3 sources
    # rather than returning an empty research dataset.
    # --------------------------------------------------------

    if not approved_sources:

        approved_sources = (
            evaluated_sources[
                :min(
                    3,
                    len(evaluated_sources)
                )
            ]
        )

    # --------------------------------------------------------
    # Summary
    # --------------------------------------------------------

    print()
    print(
        "=" * 70
    )

    print(
        "🔍 FINAL CRITIC RESULTS"
    )

    print(
        "=" * 70
    )

    print(
        f"Sources received       : "
        f"{len(sources)}"
    )

    print(
        f"Unique sources         : "
        f"{len(unique_sources)}"
    )

    print(
        f"LLM evaluations        : "
        f"{llm_count}"
    )

    print(
        f"Fallback evaluations   : "
        f"{fallback_count}"
    )

    print(
        f"Sources approved       : "
        f"{len(approved_sources)}"
    )

    print(
        f"Maximum allowed        : "
        f"{max_sources}"
    )

    print(
        "=" * 70
    )

    # --------------------------------------------------------
    # Approved source details
    # --------------------------------------------------------

    print(
        "\n🏆 APPROVED SOURCES\n"
    )

    for i, source in enumerate(
        approved_sources,
        start=1
    ):

        print(
            f"{i}. "
            f"[{source.get('source_type', 'web')}] "
            f"{source.get('title', 'Untitled')}"
        )

        print(
            f"   Score       : "
            f"{source.get('critic_score', 0):.1f}/10"
        )

        print(
            f"   Relevance   : "
            f"{source.get('critic_relevance', 0):.1f}/5"
        )

        print(
            f"   Credibility : "
            f"{source.get('critic_credibility', 0):.1f}/5"
        )

        print(
            f"   Evidence    : "
            f"{source.get('critic_evidence_quality', 0):.1f}/5"
        )

        print(
            f"   Method      : "
            f"{source.get('critic_method', 'Unknown')}"
        )

        print(
            f"   Reason      : "
            f"{source.get('critic_reason', '')[:160]}"
        )

        print()

    print(
        "=" * 70
    )

    return approved_sources


print(
    "✅ Final Hybrid Critic Agent ready."
)

✅ Final Hybrid Critic Agent ready.


In [25]:
# ============================================================
# CELL 9 — SYNTHESIZER AGENT
# ============================================================

def synthesizer_agent(
    user_query,
    subquestions,
    approved_sources
):
    """
    Generate the final research report from the approved sources.
    """

    # --------------------------------------------------------
    # Validate
    # --------------------------------------------------------

    if not user_query or not user_query.strip():
        return "❌ Research question is empty."

    if not approved_sources:
        return "❌ No reliable sources were available for synthesis."

    # --------------------------------------------------------
    # Number the sources
    # --------------------------------------------------------

    numbered_sources = []

    for i, source in enumerate(
        approved_sources,
        start=1
    ):
        numbered_sources.append({
            "id": i,
            "source_type": source.get(
                "source_type",
                "web"
            ),
            "title": source.get(
                "title",
                "Untitled Source"
            ),
            "url": source.get(
                "url",
                ""
            ),
            "content": source.get(
                "content",
                ""
            )
        })

    # --------------------------------------------------------
    # Prepare source information for the LLM
    # --------------------------------------------------------

    source_blocks = []

    for source in numbered_sources:

        source_block = f"""
SOURCE {source['id']}
Type: {source['source_type']}
Title: {source['title']}
URL: {source['url']}

Content:
{source['content'][:5000]}
"""

        source_blocks.append(source_block)

    source_packet = "\n".join(source_blocks)

    # --------------------------------------------------------
    # Prepare sub-questions
    # --------------------------------------------------------

    subquestion_text = "\n".join(
        f"{i}. {question}"
        for i, question in enumerate(
            subquestions,
            start=1
        )
    )

    # --------------------------------------------------------
    # Synthesis prompt
    # --------------------------------------------------------

    prompt = f"""
You are the final synthesis agent of an Agentic Research Assistant.

USER RESEARCH QUESTION:
{user_query}

RESEARCH SUB-QUESTIONS:
{subquestion_text}

RESEARCH SOURCES:
{source_packet}

Your task is to create a clear, structured,
evidence-based research report.

IMPORTANT RULES:

1. Use ONLY information contained in the provided sources.
2. Do not invent facts.
3. Do not invent sources.
4. Do not invent URLs.
5. Do not invent authors or paper titles.
6. Do not cite a source that was not provided.
7. Use inline citations such as [1], [2], [3].
8. Citation numbers must correspond exactly to SOURCE numbers.
9. Put citations close to the claims they support.
10. If evidence is limited or conflicting, say so clearly.
11. Do not create a References section.
12. Do not add unsupported information.

Use this structure:

# Research Title

## Abstract

Give a concise 3–4 sentence overview.

## Key Findings

Present the major findings organized around
the research sub-questions.

Use inline citations.

## Analysis

Compare and synthesize the evidence from the sources.

Use inline citations.

## Challenges and Limitations

Discuss limitations, uncertainties, conflicting evidence,
and research gaps supported by the sources.

Use inline citations where appropriate.

## Conclusion

Give a concise overall conclusion.

Remember:
Only use citation numbers that correspond to the
provided sources.
"""

    # --------------------------------------------------------
    # Call Groq
    # --------------------------------------------------------

    try:

        response = groq_client.chat.completions.create(
            model=GROQ_MODEL,
            messages=[
                {
                    "role": "system",
                    "content": (
                        "You are a careful academic research "
                        "synthesis assistant. Accuracy and "
                        "source-grounding are more important "
                        "than creativity."
                    )
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            temperature=0.2,
            max_tokens=5000
        )

        report = response.choices[0].message.content.strip()

    except Exception as e:

        print("❌ Synthesizer failed.")
        print("Error:", e)

        return f"""
# Research Assistant

## Synthesis Error

The final synthesis could not be generated.

**Error:** `{str(e)}`
"""

    # --------------------------------------------------------
    # Remove any References section generated by the LLM
    # --------------------------------------------------------

    report = re.sub(
        r"\n#+\s*References[\s\S]*$",
        "",
        report,
        flags=re.IGNORECASE
    ).strip()

    # --------------------------------------------------------
    # Validate citation numbers
    # --------------------------------------------------------

    valid_ids = {
        source["id"]
        for source in numbered_sources
    }

    citation_numbers = re.findall(
        r"\[(\d+)\]",
        report
    )

    for number in citation_numbers:

        number_int = int(number)

        if number_int not in valid_ids:

            report = report.replace(
                f"[{number}]",
                ""
            )

    # --------------------------------------------------------
    # Generate REAL References section
    # --------------------------------------------------------

    references = "\n\n## References\n\n"

    for source in numbered_sources:

        references += (
            f"**[{source['id']}]** "
            f"{source['title']}  \n"
            f"{source['url']}\n\n"
        )

    # --------------------------------------------------------
    # Final report
    # --------------------------------------------------------

    final_report = (
        report
        + references
    )

    return final_report


print("✅ Synthesizer Agent ready.")

✅ Synthesizer Agent ready.


In [26]:
# ============================================================
# CELL 10 — MASTER RESEARCH PIPELINE
# ============================================================

def run_research_assistant(user_query):
    """
    Complete Agentic Research Assistant pipeline.

    Flow:
    User Question
        ↓
    Planner
        ↓
    Searcher
        ↓
    Reader
        ↓
    Critic
        ↓
    Synthesizer
        ↓
    Final Report
    """

    # --------------------------------------------------------
    # Validate input
    # --------------------------------------------------------

    if not user_query or not user_query.strip():
        return "❌ Please enter a research question."

    user_query = user_query.strip()

    print("=" * 70)
    print("🤖 AGENTIC RESEARCH ASSISTANT")
    print("=" * 70)

    print("\n📝 Research Question:")
    print(user_query)

    # ========================================================
    # STEP 1 — PLANNER
    # ========================================================

    print("\n" + "=" * 70)
    print("🧠 STEP 1 — PLANNER")
    print("=" * 70)

    subquestions = planner_agent(user_query)

    print("\nGenerated Sub-Questions:")

    for i, question in enumerate(subquestions, start=1):
        print(f"{i}. {question}")

    # ========================================================
    # STEP 2 — SEARCHER
    # ========================================================

    print("\n" + "=" * 70)
    print("🔎 STEP 2 — SEARCHER")
    print("=" * 70)

    raw_sources = searcher_agent(subquestions)

    print(
        f"\n📊 Raw sources retrieved: {len(raw_sources)}"
    )

    # ========================================================
    # STEP 3 — READER
    # ========================================================

    print("\n" + "=" * 70)
    print("📖 STEP 3 — READER")
    print("=" * 70)

    sources = reader_agent(raw_sources)

    print(
        f"📚 Sources after reading/normalization: {len(sources)}"
    )

    # ========================================================
    # STEP 4 — CRITIC
    # ========================================================

    print("\n" + "=" * 70)
    print("🔍 STEP 4 — CRITIC")
    print("=" * 70)

    approved_sources = critic_filter(
        sources,
        max_sources=12
    )

    print(
        f"\n✅ Sources approved for synthesis: "
        f"{len(approved_sources)}"
    )

    # --------------------------------------------------------
    # No usable sources
    # --------------------------------------------------------

    if not approved_sources:

        return f"""
# Research Assistant

## ⚠️ No Reliable Sources Found

I couldn't find enough usable sources for:

**{user_query}**

Please try a more specific research question.
"""

    # ========================================================
    # STEP 5 — SYNTHESIZER
    # ========================================================

    print("\n" + "=" * 70)
    print("✍️ STEP 5 — SYNTHESIZER")
    print("=" * 70)

    final_report = synthesizer_agent(
        user_query=user_query,
        subquestions=subquestions,
        approved_sources=approved_sources
    )

    # ========================================================
    # COMPLETE
    # ========================================================

    print("\n" + "=" * 70)
    print("🎉 RESEARCH COMPLETE")
    print("=" * 70)

    return final_report


print("✅ Master Research Pipeline ready.")

✅ Master Research Pipeline ready.


In [27]:
# ============================================================
# CELL 11 — COMPLETE BACKEND TEST
# ============================================================

test_query = "How is artificial intelligence transforming healthcare?"

print("🚀 Starting complete backend test...")
print()

test_result = run_research_assistant(test_query)

print("\n")
print("=" * 80)
print("📄 FINAL RESEARCH REPORT")
print("=" * 80)
print()

print(test_result)

🚀 Starting complete backend test...

🤖 AGENTIC RESEARCH ASSISTANT

📝 Research Question:
How is artificial intelligence transforming healthcare?

🧠 STEP 1 — PLANNER

Generated Sub-Questions:
1. Which AI-driven tools are currently being used to improve diagnostic accuracy and treatment planning in clinical practice, and what evidence supports their effectiveness?
2. What ethical, regulatory, and data privacy challenges arise from implementing AI systems in patient care, and how are they being addressed?
3. How is the adoption of AI technologies affecting healthcare workforce roles, training needs, and cost efficiency across different care settings?

🔎 STEP 2 — SEARCHER

🔎 RESEARCHING SUB-QUESTION 1/3
Which AI-driven tools are currently being used to improve diagnostic accuracy and treatment planning in clinical practice, and what evidence supports their effectiveness?
🌐 Tavily results: 3
📚 arXiv results: 1

🔎 RESEARCHING SUB-QUESTION 2/3
What ethical, regulatory, and data privacy challen

In [47]:
# ============================================================
# CELL 12 — USER INPUT
# ============================================================

print("=" * 70)
print("🤖 AGENTIC RESEARCH ASSISTANT")
print("=" * 70)
print()
print("Enter the research topic you want to investigate.")
print("Type 'exit' to stop.")
print()

user_query = input("🔎 Your research question: ").strip()

if user_query.lower() == "exit":
    print("👋 Research session ended.")

elif not user_query:
    print("❌ Please enter a valid research question.")

else:
    print()
    print("🚀 Starting research...")
    print()

    final_report = run_research_assistant(user_query)

    print()
    print("=" * 80)
    print("📄 FINAL RESEARCH REPORT")
    print("=" * 80)
    print()

    print(final_report)

🤖 AGENTIC RESEARCH ASSISTANT

Enter the research topic you want to investigate.
Type 'exit' to stop.

🔎 Your research question: articles about mechanical engineering

🚀 Starting research...

🤖 AGENTIC RESEARCH ASSISTANT

📝 Research Question:
articles about mechanical engineering

🧠 STEP 1 — PLANNER

Generated Sub-Questions:
1. What are the predominant research themes and subject areas covered in recent mechanical engineering articles across leading journals and conferences?
2. How do publication metrics (impact factor, citation counts, open‑access availability) vary among top mechanical engineering journals, and what trends can be observed over the past decade?
3. Which emerging technologies and interdisciplinary approaches are most frequently highlighted in contemporary mechanical engineering literature, and how are they shaping future research directions?

🔎 STEP 2 — SEARCHER

🔎 RESEARCHING SUB-QUESTION 1/3
What are the predominant research themes and subject areas covered in recent 